# 第4周｜Logistic分类与概率

学习目标：logit、OR、阈值、混淆矩阵、AUROC、AUPRC。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：解释OR、预测概率和分类标签三个概念；阈值0.5仅教学。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## Logistic回归：概率不等于标签
本周预测的是已有的基线糖尿病状态，不是将来的糖尿病发病。先排除与诊断密切相关的HbA1c，只用年龄、BMI、吸烟练习。OR不是概率差，也不是风险比。

In [2]:
import statsmodels.api as sm
from primecvd.basics import design_basics
X = design_basics(train)
model = sm.GLM(train.diabetes, X, family=sm.families.Binomial()).fit()
interval = model.conf_int()
display(pd.DataFrame({"OR":np.exp(model.params), "CI_low":np.exp(interval[0]), "CI_high":np.exp(interval[1])}))
probability = model.predict(design_basics(valid)).to_numpy()
print("前五名验证人员的预测概率：", probability[:5])

,OR,CI_low,CI_high
intercept,0.000156,0.000111,0.000219
Age_per10,1.784998,1.723923,1.848236
BMI_per5,1.709233,1.638972,1.782507
smoking_ex,1.007048,0.901121,1.125426
smoking_current,0.967424,0.841790,1.111809


前五名验证人员的预测概率： [0.02968007 0.05254953 0.01649575 0.01924644 0.13567937]


In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
print("AUROC：", roc_auc_score(valid.diabetes, probability))
print("AUPRC：", average_precision_score(valid.diabetes, probability))
print("结局比例（AUPRC参考）：", valid.diabetes.mean())
print("Brier：", brier_score_loss(valid.diabetes, probability))
display(pd.DataFrame(confusion_matrix(valid.diabetes, probability >= 0.5, labels=[0,1]),
                     index=["实际0", "实际1"], columns=["预测0", "预测1"]))

AUROC： 0.7368766599323396
AUPRC： 0.2218670374666344
结局比例（AUPRC参考）： 0.07426666666666666
Brier： 0.06378756064608618


,预测0,预测1
实际0,6940,3
实际1,553,4


阈值0.5只是事先固定的教学示例，不是临床推荐阈值。模型即使能把高风险者排在前面，也可能在这个阈值下几乎不报阳性。不要靠不断查看测试集选阈值。

## 独立练习
分别用一句话解释：OR=2、预测概率=0.2、分类标签=0。观察类别比例，讨论只报告准确率的问题。说明为什么不能对有早期删失的CVD结局照搬这段二分类代码。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：横断面分类不是未来糖尿病发病预测。